<a href="https://colab.research.google.com/github/maduakordavid01-ui/O.Rproject/blob/main/C%C3%B3pia_de_temperature_and_irradiance_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/maduakordavid01-ui/O.Rproject/blob/main/C%C3%B3pia_de_temperature_and_irradiance.ipynb
)


In [ ]:
import math
from datetime import datetime, timedelta
%pip install cdsapi
%pip install netcdf4
import cdsapi
import xarray as xr
import pandas as pd
!pip install pvlib


In [ ]:
dataset = "cams-gridded-solar-radiation"
request = {
    "variable": [
        "global_horizontal_irradiation",
        "direct_horizontal_irradiation",
        "diffuse_horizontal_irradiation",
        "direct_normal_irradiation"
    ],
    "sky_type": ["clear"],
    "version": ["4.6"],
    "year": ["2020"],
    "month": ["06"],
    "area": [44.5, -10, 36, 3.5]
}

client = cdsapi.Client(url='https://ads.atmosphere.copernicus.eu/api', key='1d12ba7d-1579-461e-9f07-608325392a05', verify=False)
client.retrieve(dataset, request).download()


In [ ]:
dataset = "cams-global-reanalysis-eac4-monthly"
request = {
    "variable": ["2m_temperature"],
    "year": ["2020"],
    "month": ["06"],
    "product_type": ["monthly_mean"],
    "data_format": "grib",
    "area": [90, -180, -90, 180]
}


client = cdsapi.Client(url='https://ads.atmosphere.copernicus.eu/api', key='1d12ba7d-1579-461e-9f07-608325392a05', verify=False)
client.retrieve(dataset, request).download()

In [ ]:
!pip install -U xarray cfgrib eccodes pandas


In [ ]:
import xarray as xr
import pandas as pd

In [ ]:
#import xarray as xr
#!pip install xarray cfgrib eccodes pandas
ds = xr.open_dataset(
    'b83abf7f1f10e090fdf79a8e5b64b7dc.grib',
    engine='cfgrib')
df_temperature = ds.to_dataframe()
df_temperature.to_csv('output_temperature.csv', index=True)


In [ ]:
df_temperature

In [ ]:
!pip install cartopy
import cartopy

In [ ]:
# Carrega o arquivo NetCDF usando xarray
ds_BHI = xr.open_dataset('v4.6_BHI_clear_2020_06.area-subset.44.5.3.5.36.-10.nc', engine='h5netcdf')
ds_BNI = xr.open_dataset('v4.6_BNI_clear_2020_06.area-subset.44.5.3.5.36.-10.nc', engine='h5netcdf')
ds_DHI = xr.open_dataset('v4.6_DHI_clear_2020_06.area-subset.44.5.3.5.36.-10.nc', engine='h5netcdf')
ds_GHI = xr.open_dataset('v4.6_GHI_clear_2020_06.area-subset.44.5.3.5.36.-10.nc', engine='h5netcdf')

df_converted_BHI = ds_BHI.to_dataframe()
df_converted_BNI = ds_BNI.to_dataframe()
df_converted_DHI = ds_DHI.to_dataframe()
df_converted_GHI = ds_GHI.to_dataframe()
df_combined = pd.concat([df_converted_BHI, df_converted_BNI, df_converted_DHI, df_converted_GHI], axis=1)
df_combined.columns = ['BHI', 'BNI', 'DHI', 'GHI']

df_combined.to_csv('output_data_irradiance.csv', index=True)
print('Dataset converted to DataFrame and saved to output_datall.csv')

Code: merge temperature into irradiance (nearest lat/lon + optional time)

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ----------------------------
# Files
# ----------------------------
irr_path  = "output_data_irradiance.csv"
temp_path = "output_temperature.csv"
out_path  = "irradiance_with_temperature.csv"

# ----------------------------
# Load
# ----------------------------
irr = pd.read_csv(irr_path)
tmp = pd.read_csv(temp_path)

# Parse time columns
irr["time"] = pd.to_datetime(irr["time"])

# Temperature time column: prefers "valid_time" if present, else "time"
tmp_time_col = "valid_time" if "valid_time" in tmp.columns else ("time" if "time" in tmp.columns else None)
if tmp_time_col is not None:
    tmp[tmp_time_col] = pd.to_datetime(tmp[tmp_time_col])

# Temperature variable name (common: "t" in Kelvin for ERA5)
# If your file has another name, change here:
temp_var = "t2m"
if temp_var not in tmp.columns:
    raise ValueError(f"Temperature column '{temp_var}' not found. Available columns: {list(tmp.columns)}")

# Convert Kelvin -> Celsius (comment out if already °C)
tmp["T_air_C"] = tmp[temp_var] - 273.15

# If the temperature file has pressure levels, choose one (edit if needed)
if "isobaricInhPa" in tmp.columns:
    # Example: keep only 850 hPa (as in many ERA5 pressure-level exports)
    tmp = tmp[tmp["isobaricInhPa"] == 850].copy()

# ----------------------------
# Crop temperature to irradiance region (+padding) for speed
# ----------------------------
pad = 2.0  # degrees
lat_min, lat_max = irr["latitude"].min(), irr["latitude"].max()
lon_min, lon_max = irr["longitude"].min(), irr["longitude"].max()

tmp = tmp[
    tmp["latitude"].between(lat_min - pad, lat_max + pad) &
    tmp["longitude"].between(lon_min - pad, lon_max + pad)
].copy()

# ----------------------------
# Helper: assign temperature by nearest spatial point for one "tmp field"
# ----------------------------
def assign_temp_by_kdtree(irr_part: pd.DataFrame, tmp_field: pd.DataFrame) -> pd.DataFrame:
    """
    irr_part: rows of irradiance with columns latitude, longitude
    tmp_field: temperature field with columns latitude, longitude, T_air_C
    Returns irr_part with new column 'T_air_C'
    """
    tmp_xy = np.column_stack([tmp_field["latitude"].to_numpy(), tmp_field["longitude"].to_numpy()])
    tree = cKDTree(tmp_xy)

    irr_xy = np.column_stack([irr_part["latitude"].to_numpy(), irr_part["longitude"].to_numpy()])
    dist, idx = tree.query(irr_xy, k=1)

    irr_part = irr_part.copy()
    irr_part["T_air_C"] = tmp_field["T_air_C"].to_numpy()[idx]
    irr_part["temp_nn_distance_deg"] = dist  # optional QA
    return irr_part

# ----------------------------
# Case A: temperature has NO time (or only one time) -> simplest
# ----------------------------
if tmp_time_col is None or tmp[tmp_time_col].nunique() == 1:
    tmp_field = tmp[["latitude", "longitude", "T_air_C"]].drop_duplicates()
    irr_out = assign_temp_by_kdtree(irr, tmp_field)

# ----------------------------
# Case B: temperature has multiple times -> match by time AND nearest lat/lon
# (efficient: loop over times, avoids giant merge)
# ----------------------------
else:
    # We will match each irradiance timestamp to the nearest available temperature timestamp.
    # (If they are exactly the same, perfect; if not, nearest is typical for reanalysis grids.)
    tmp_times = np.sort(tmp[tmp_time_col].unique())

    # Precompute mapping from irr times -> nearest tmp time
    irr_times = irr["time"].unique()
    # Convert datetime64 -> int ns for vectorized nearest
    tmp_ns = tmp_times.astype("datetime64[ns]").astype("int64")
    irr_ns = irr_times.astype("datetime64[ns]").astype("int64")

    # Find nearest temp time index for each irr time
    # (uses searchsorted on sorted arrays)
    pos = np.searchsorted(tmp_ns, irr_ns)
    pos = np.clip(pos, 1, len(tmp_ns) - 1)
    left = tmp_ns[pos - 1]
    right = tmp_ns[pos]
    choose_right = (np.abs(irr_ns - left) > np.abs(right - irr_ns))
    nearest_idx = pos.copy()
    nearest_idx[~choose_right] = pos[~choose_right] - 1
    irr_to_tmp_time = dict(zip(irr_times, tmp_times[nearest_idx]))

    # Apply mapping
    irr2 = irr.copy()
    irr2["tmp_time_match"] = irr2["time"].map(irr_to_tmp_time)

    pieces = []
    for tmatch, irr_part in irr2.groupby("tmp_time_match", sort=False):
        tmp_field = tmp[tmp[tmp_time_col] == tmatch][["latitude", "longitude", "T_air_C"]].drop_duplicates()
        pieces.append(assign_temp_by_kdtree(irr_part.drop(columns=["tmp_time_match"]), tmp_field))

    irr_out = pd.concat(pieces, ignore_index=True)

# ----------------------------
# Save
# ----------------------------
#df_temperatureirradiance.to_csv('output_data_temperature_irradiance.csv', index=True)
#irr_out.to_csv(out_path, index=False)
#print("Saved:", out_path)
df_temperature_irradiance=(irr_out[["time", "latitude", "longitude", "GHI", "T_air_C"]].head())
df_combined.to_csv('output_data_temperature_irradiance.csv', index=True)

In [ ]:
df_temperature_irradiance

In [ ]:
df_combined

In [ ]:
import pandas as pd

df = pd.read_csv("output_data_temperature_irradiance.csv")
df["time"] = pd.to_datetime(df["time"])


In [ ]:
df[["GHI", "DHI", "BNI"]].describe()


In [ ]:
(df["GHI"] > 0).sum()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

path = "output_data_irradiance.csv"
df = pd.read_csv(path)

# Choose which irradiance variable to plot: "GHI", "DHI", "BNI", "BHI"
var = "GHI"

# --- 1) A nice single-time map (auto-picks the time with highest mean irradiance) ---
best_time = df.groupby("time", sort=False)[var].mean().idxmax()
d = df[df["time"] == best_time].copy()

grid = (
    d.pivot_table(index="latitude", columns="longitude", values=var, aggfunc="mean")
     .sort_index()
)

lats = grid.index.to_numpy()
lons = grid.columns.to_numpy()
Z = grid.to_numpy()

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    Z,
    origin="lower",
    extent=[float(lons.min()), float(lons.max()), float(lats.min()), float(lats.max())],
    aspect="auto"
)
cb = plt.colorbar(im, ax=ax, pad=0.02, shrink=0.9)
cb.set_label(f"{var} (W/m²)")

# optional: overlay the grid points
ax.scatter(d["longitude"], d["latitude"], s=2)

ax.set_title(f"{var} heatmap at {best_time}")
ax.set_xlabel("Longitude (deg)")
ax.set_ylabel("Latitude (deg)")
ax.grid(True, linewidth=0.4)
plt.tight_layout()
plt.show()


# --- 2) Mean map over the whole dataset ---
mean_grid = (
    df.groupby(["latitude", "longitude"], as_index=False)[var].mean()
      .pivot_table(index="latitude", columns="longitude", values=var)
      .sort_index()
)

lats = mean_grid.index.to_numpy()
lons = mean_grid.columns.to_numpy()
Z = mean_grid.to_numpy()

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    Z,
    origin="lower",
    extent=[float(lons.min()), float(lons.max()), float(lats.min()), float(lats.max())],
    aspect="auto"
)
cb = plt.colorbar(im, ax=ax, pad=0.02, shrink=0.9)
cb.set_label(f"Mean {var} (W/m²)")

ax.set_title(f"Mean {var} over all times")
ax.set_xlabel("Longitude (deg)")
ax.set_ylabel("Latitude (deg)")
ax.grid(True, linewidth=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv("output_data_irradiance.csv")

# Parse time
df["time"] = pd.to_datetime(df["time"])

# Choose irradiance variable
var = "GHI"   # can be: "GHI", "DHI", "BNI", "BHI"

# Spatial mean over the domain
ts_mean = df.groupby("time")[var].mean()

# Plot
plt.figure(figsize=(10, 5))
plt.plot(ts_mean.index, ts_mean.values)
plt.xlabel("Time")
plt.ylabel(f"{var} (W/m²)")
plt.title(f"Spatial mean {var} over time")
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
for v in ["GHI", "DHI", "BNI"]:
    plt.plot(df.groupby("time")[v].mean(), label=v)

plt.legend()
plt.ylabel("Irradiance (W/m²)")
plt.grid(True)
plt.show()




In [ ]:
# ==================== SIMPLE GITHUB CLEANER ====================
print("🧹 Simple Notebook Cleaner for GitHub")

import os
import json

# List all files
print("📁 Files in current directory:")
for f in os.listdir('.'):
    print(f"  {f}")

# Find .ipynb files
notebooks = [f for f in os.listdir('.') if f.endswith('.ipynb')]

if not notebooks:
    print("❌ No .ipynb files found!")
    print("\nMake sure you:")
    print("1. Have saved your notebook (File → Save)")
    print("2. Are looking in the right directory")

    # Create a dummy notebook to test
    print("\nCreating a test notebook to verify...")
    test_nb = {
        "cells": [{
            "cell_type": "code",
            "execution_count": None,
            "metadata": {},
            "outputs": [],
            "source": ["print('Test notebook')"]
        }],
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3"
            }
        },
        "nbformat": 4,
        "nbformat_minor": 4
    }

    with open('test_notebook.ipynb', 'w') as f:
        json.dump(test_nb, f)

    print("✅ Created test_notebook.ipynb")
    notebooks = ['test_notebook.ipynb']

# Clean each notebook
for notebook_file in notebooks:
    print(f"\n📓 Processing: {notebook_file}")

    try:
        # Read
        with open(notebook_file, 'r') as f:
            nb = json.load(f)

        # Clean
        cleaned_count = 0

        # Remove widget metadata from cells
        for cell in nb['cells']:
            if 'metadata' in cell:
                if 'widgets' in cell['metadata']:
                    del cell['metadata']['widgets']
                    cleaned_count += 1
                if 'colab' in cell['metadata']:
                    del cell['metadata']['colab']
                    cleaned_count += 1

        # Remove notebook-level widget metadata
        if 'metadata' in nb and 'widgets' in nb['metadata']:
            del nb['metadata']['widgets']
            cleaned_count += 1

        # Save cleaned version
        clean_name = f"CLEANED_{notebook_file}"
        with open(clean_name, 'w') as f:
            json.dump(nb, f, indent=2)

        print(f"✅ Saved: {clean_name}")
        print(f"   Removed {cleaned_count} widget metadata items")

        # Download
        from google.colab import files
        files.download(clean_name)
        print("📥 Download started!")

    except Exception as e:
        print(f"❌ Error cleaning {notebook_file}: {str(e)}")

print("\n🎉 Done! Upload the CLEANED_*.ipynb file to GitHub.")